# EMA + RSI

EMA Crossover + RSI Filter \
It uses ATR(14) for dynamic stops/tolerance (adapts to volatility). \
Momentum strategy, excellent for scalping and intraday across all TFs. \
Filters false signals with RSI.

__How EMA+RSI Algorithm Determines Entry/Exit:__
- Fast EMA (9) / Slow EMA (21) – standard for crypto.
- Long Entry: Fast EMA crosses above Slow EMA AND RSI(14) < 70 (not overbought).
- Short Entry: Fast EMA crosses below Slow EMA AND RSI(14) > 30 (not oversold).
- Exit: Reverse crossover OR price hits ATR-based trailing stop.
- Excellent filter reduces whipsaws in ranging markets.

## Configuration

In [1]:
# --- path bootstrap: make 'entry_exit_points' importable from any CWD ---
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / "entry_exit_points" / "__init__.py").exists():
    if root == root.parent: raise RuntimeError("project root not found")
    root = root.parent
if str(root) not in sys.path: sys.path.insert(0, str(root))

In [2]:
from entry_exit_points.fetcher import BybitFetcher
from entry_exit_points.backtester import Backtester
from entry_exit_points.models import StrategyConfig, Signal, SignalAction
from entry_exit_points.visualization import build_chart
from IPython.display import HTML, display

In [3]:
SYMBOL   = "BTCUSDT"
INTERVAL = "15"
CANDLES  = 800

In [4]:
# Fetch data
fetcher = BybitFetcher()
df = fetcher.fetch_klines(
    symbol=SYMBOL,
    interval=INTERVAL,
    num_candles=CANDLES,
    start_time="2026-03-20",
    end_time="2026-04-19",    # optional; defaults to now
)
fetcher.close()

## EMA + RSI

In [5]:
# Import EMA + RSI strategy
from entry_exit_points.strategies import EMACrossoverStrategy

In [6]:
# Backtest EMA + RSI strategy
config = StrategyConfig()
strategy = EMACrossoverStrategy(config)
bt = Backtester(strategy, symbol=SYMBOL)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

════════════════════════════════════════════════════════════
  Backtest Summary: ema
  BTCUSDT | 15 | 2881 bars
════════════════════════════════════════════════════════════
  Total trades      : 103
  Win / Loss         : 27 / 76
  Win rate           : 26.2%
  Total P&L (bps)    : -736.6
  Avg P&L (bps)      : -7.2
  Max win (bps)      : +467.5
  Max loss (bps)     : -117.1
  Profit factor      : 0.79
  Max drawdown (bps) : 960.3
  Sharpe (approx)    : -0.08
════════════════════════════════════════════════════════════


In [ ]:
# EMA + RSI strategy chart
prepared = strategy.prepare(df)
signals = []
for t in result.trades:
    if t.entry_ts:
        signals.append(Signal(t.entry_ts, SignalAction.ENTRY, t.direction, t.entry_price))
    if t.exit_ts:
        signals.append(Signal(t.exit_ts, SignalAction.EXIT, t.direction, t.exit_price))

# build_chart saves to HTML — to display inline instead:
chart_file = f"chart_{strategy.name}.html"

build_chart(
    prepared, signals,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
    save_path=chart_file,
)

# Display inline
display(HTML(open(chart_file).read()))